# Structured Outputs: Guaranteed JSON from LLMs

LLMs are trained to generate human-readable text. When you ask an LLM to "return JSON", it does its best, but it was not trained to guarantee schema compliance. In production systems, a single malformed response can crash a data pipeline, corrupt a database, or surface a traceback to the user.

This notebook covers three progressively more reliable approaches:
1. **Fragile approach**: Prompt engineering + manual parsing
2. **Pydantic validation**: Schema enforcement at parse time
3. **instructor library**: Automatic retry with validation feedback

We also cover Anthropic tool use and OpenAI JSON mode as API-level alternatives.

In [1]:
import json
import re
import time
from enum import Enum
from typing import Optional
from pydantic import BaseModel, ValidationError, field_validator

# instructor is a library that wraps Anthropic/OpenAI clients to add structured output
try:
    import instructor
    print(f'instructor version: {instructor.__version__}')
except ImportError:
    print('instructor not installed. Run: pip install instructor')

print('Pydantic version:', __import__('pydantic').__version__)
print('Note: This notebook uses mock responses where API keys are not available.')

instructor version: 1.15.4
Pydantic version: 2.13.4
Note: This notebook uses mock responses where API keys are not available.


## The Problem: What LLMs Actually Return When You Ask for JSON

You ask for JSON. Here is what you might get back:

In [2]:
# These are all real examples of what LLMs return when prompted to "return JSON"

RESPONSE_VARIANTS = [
    # Ideal response
    '{"sentiment": "positive", "score": 8, "summary": "Great product"}',
    
    # With markdown code fences (extremely common)
    '```json\n{"sentiment": "positive", "score": 8, "summary": "Great product"}\n```',
    
    # With prose before the JSON
    'Here is the JSON you requested:\n{"sentiment": "positive", "score": 8, "summary": "Great product"}',
    
    # With trailing explanation
    '{"sentiment": "positive", "score": 8, "summary": "Great product"}\n\nNote: score is out of 10.',
    
    # Wrong schema: missing field
    '{"sentiment": "positive", "score": 8}',
    
    # Wrong type: score as string instead of int
    '{"sentiment": "positive", "score": "eight", "summary": "Great product"}',
    
    # Completely off-format
    'The sentiment is positive with a score of 8 out of 10. Summary: Great product.',
]

print('LLM response variants when asked for JSON:')
for i, r in enumerate(RESPONSE_VARIANTS):
    print(f'\nVariant {i+1}: {repr(r[:80])}')
print('\nOnly variant 1 parses cleanly. The rest require special handling.')

LLM response variants when asked for JSON:

Variant 1: '{"sentiment": "positive", "score": 8, "summary": "Great product"}'

Variant 2: '```json\n{"sentiment": "positive", "score": 8, "summary": "Great product"}\n```'

Variant 3: 'Here is the JSON you requested:\n{"sentiment": "positive", "score": 8, "summary":'

Variant 4: '{"sentiment": "positive", "score": 8, "summary": "Great product"}\n\nNote: score i'

Variant 5: '{"sentiment": "positive", "score": 8}'

Variant 6: '{"sentiment": "positive", "score": "eight", "summary": "Great product"}'

Variant 7: 'The sentiment is positive with a score of 8 out of 10. Summary: Great product.'

Only variant 1 parses cleanly. The rest require special handling.


## Section 1: The Fragile Approach (What Not to Do)

The naive approach: prompt the model to return JSON, then parse with `json.loads`. This fails on 6 out of 7 variants above.

In [3]:
# The fragile approach: try to parse, hope for the best

def fragile_parse(llm_response: str) -> dict:
    """Naive approach: just try json.loads. Fails often."""
    return json.loads(llm_response)  # Will raise JSONDecodeError on most variants

print('Testing fragile_parse on each variant:')
success = 0
failure = 0
for i, response in enumerate(RESPONSE_VARIANTS):
    try:
        result = fragile_parse(response)
        print(f'Variant {i+1}: SUCCESS -> {result}')
        success += 1
    except (json.JSONDecodeError, Exception) as e:
        print(f'Variant {i+1}: FAILURE -> {type(e).__name__}: {str(e)[:60]}')
        failure += 1

print(f'\nResult: {success} successes, {failure} failures out of {len(RESPONSE_VARIANTS)} variants')
print('This is NOT production-ready.')

Testing fragile_parse on each variant:
Variant 1: SUCCESS -> {'sentiment': 'positive', 'score': 8, 'summary': 'Great product'}
Variant 2: FAILURE -> JSONDecodeError: Expecting value: line 1 column 1 (char 0)
Variant 3: FAILURE -> JSONDecodeError: Expecting value: line 1 column 1 (char 0)
Variant 4: FAILURE -> JSONDecodeError: Extra data: line 3 column 1 (char 67)
Variant 5: SUCCESS -> {'sentiment': 'positive', 'score': 8}
Variant 6: SUCCESS -> {'sentiment': 'positive', 'score': 'eight', 'summary': 'Great product'}
Variant 7: FAILURE -> JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Result: 3 successes, 4 failures out of 7 variants
This is NOT production-ready.


In [4]:
# A somewhat better manual approach: regex to extract JSON
# Still fragile, but handles markdown fences

def manual_parse_attempt(llm_response: str) -> dict:
    """
    Attempt to extract and parse JSON from LLM response.
    Handles markdown fences and some leading text.
    Still fragile -- does not validate schema.
    """
    # Remove markdown fences
    cleaned = re.sub(r'^```(?:json)?\s*', '', llm_response.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r'```\s*$', '', cleaned.strip(), flags=re.MULTILINE)
    cleaned = cleaned.strip()
    
    # Try to find a JSON object with regex
    json_match = re.search(r'\{[^{}]*\}', cleaned, re.DOTALL)
    if json_match:
        cleaned = json_match.group()
    
    return json.loads(cleaned)

print('Testing manual_parse_attempt on each variant:')
success = 0
failure = 0
for i, response in enumerate(RESPONSE_VARIANTS):
    try:
        result = manual_parse_attempt(response)
        print(f'Variant {i+1}: SUCCESS -> {result}')
        success += 1
    except (json.JSONDecodeError, Exception) as e:
        print(f'Variant {i+1}: FAILURE -> {type(e).__name__}: {str(e)[:60]}')
        failure += 1

print(f'\nResult: {success} successes, {failure} failures')
print('Better, but still fails on wrong types and missing fields.')
print('\nThe core problem: parsing succeeds but schema validation fails silently.')

Testing manual_parse_attempt on each variant:
Variant 1: SUCCESS -> {'sentiment': 'positive', 'score': 8, 'summary': 'Great product'}
Variant 2: SUCCESS -> {'sentiment': 'positive', 'score': 8, 'summary': 'Great product'}
Variant 3: SUCCESS -> {'sentiment': 'positive', 'score': 8, 'summary': 'Great product'}
Variant 4: SUCCESS -> {'sentiment': 'positive', 'score': 8, 'summary': 'Great product'}
Variant 5: SUCCESS -> {'sentiment': 'positive', 'score': 8}
Variant 6: SUCCESS -> {'sentiment': 'positive', 'score': 'eight', 'summary': 'Great product'}
Variant 7: FAILURE -> JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Result: 6 successes, 1 failures
Better, but still fails on wrong types and missing fields.

The core problem: parsing succeeds but schema validation fails silently.


## Section 2: Pydantic Models for Output Schema

Define the expected output schema as a Pydantic model. This catches wrong types, missing fields, and invalid values at parse time.

In [5]:
class SentimentEnum(str, Enum):
    positive = 'positive'
    negative = 'negative'
    neutral = 'neutral'


class ProductReview(BaseModel):
    """Schema for product review sentiment analysis output."""
    sentiment: SentimentEnum
    score: int  # 1-10
    summary: str
    
    @field_validator('score')
    @classmethod
    def score_must_be_in_range(cls, v: int) -> int:
        if not 1 <= v <= 10:
            raise ValueError(f'score must be between 1 and 10, got {v}')
        return v
    
    @field_validator('summary')
    @classmethod
    def summary_must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('summary cannot be empty')
        return v.strip()


def parse_with_pydantic(llm_response: str) -> ProductReview:
    """Parse LLM response and validate against Pydantic schema."""
    # Clean markdown fences
    cleaned = re.sub(r'^```(?:json)?\s*', '', llm_response.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r'```\s*$', '', cleaned.strip(), flags=re.MULTILINE)
    
    # Extract JSON object if mixed with text
    json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)
    if json_match:
        cleaned = json_match.group()
    
    data = json.loads(cleaned.strip())
    return ProductReview(**data)  # Validates schema


print('Testing Pydantic validation on each variant:')
for i, response in enumerate(RESPONSE_VARIANTS):
    try:
        result = parse_with_pydantic(response)
        print(f'Variant {i+1}: SUCCESS -> sentiment={result.sentiment.value}, score={result.score}')
    except (json.JSONDecodeError, ValidationError, Exception) as e:
        print(f'Variant {i+1}: FAILURE -> {type(e).__name__}: {str(e)[:80]}')

Testing Pydantic validation on each variant:
Variant 1: SUCCESS -> sentiment=positive, score=8
Variant 2: SUCCESS -> sentiment=positive, score=8
Variant 3: SUCCESS -> sentiment=positive, score=8
Variant 4: SUCCESS -> sentiment=positive, score=8
Variant 5: FAILURE -> ValidationError: 1 validation error for ProductReview
summary
  Field required [type=missing, inp
Variant 6: FAILURE -> ValidationError: 1 validation error for ProductReview
score
  Input should be a valid integer, un
Variant 7: FAILURE -> JSONDecodeError: Expecting value: line 1 column 1 (char 0)


In [6]:
# Show the validation error detail -- this is what you feed back to the model
try:
    bad_response = '{"sentiment": "positive", "score": "eight", "summary": "Great product"}'
    parse_with_pydantic(bad_response)
except ValidationError as e:
    print('ValidationError details (structured, machine-readable):')
    print(json.dumps(e.errors(), indent=2))
    print()
    print('Human-readable error string:')
    print(str(e))

ValidationError details (structured, machine-readable):
[
  {
    "type": "int_parsing",
    "loc": [
      "score"
    ],
    "msg": "Input should be a valid integer, unable to parse string as an integer",
    "input": "eight",
    "url": "https://errors.pydantic.dev/2.13/v/int_parsing"
  }
]

Human-readable error string:
1 validation error for ProductReview
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='eight', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## Section 3: The instructor Library

instructor wraps the Anthropic and OpenAI clients to add automatic schema enforcement. When the model returns invalid output, instructor formats the validation error and retries the request automatically.

In [7]:
# instructor with Anthropic
# In production (with API key):
#
#   import anthropic
#   import instructor
#
#   client = instructor.from_anthropic(anthropic.Anthropic())
#
#   review = client.messages.create(
#       model='claude-3-5-sonnet-20241022',
#       max_tokens=500,
#       messages=[{
#           'role': 'user',
#           'content': 'Analyze this review: "Best laptop I have ever owned! Fast, lightweight.""
#       }],
#       response_model=ProductReview,
#   )
#   # review is a validated ProductReview instance -- no parsing, no try/except
#   print(review.sentiment, review.score, review.summary)

# instructor with OpenAI
# In production (with API key):
#
#   import openai
#   import instructor
#
#   client = instructor.from_openai(openai.OpenAI())
#
#   review = client.chat.completions.create(
#       model='gpt-4o',
#       messages=[{
#           'role': 'user',
#           'content': 'Analyze this review: "Best laptop I have ever owned!"'
#       }],
#       response_model=ProductReview,
#   )

print('instructor usage patterns shown above.')
print('Key difference from raw API: response_model parameter replaces manual parsing.')
print('instructor handles: markdown removal, JSON extraction, Pydantic validation, retries.')

instructor usage patterns shown above.
Key difference from raw API: response_model parameter replaces manual parsing.
instructor handles: markdown removal, JSON extraction, Pydantic validation, retries.


In [8]:
# Simulate what instructor does internally: automatic retry on validation failure

def simulate_instructor_retry(
    mock_responses: list[str],
    model_class: type[BaseModel],
    max_retries: int = 3
) -> tuple[BaseModel, int]:
    """
    Simulate instructor's retry logic.
    In reality, instructor sends the validation error back to the LLM.
    Here we simulate with a list of mock responses.
    Returns (validated_object, attempts_needed).
    """
    last_error = None
    
    for attempt in range(1, max_retries + 1):
        response = mock_responses[min(attempt - 1, len(mock_responses) - 1)]
        print(f'\nAttempt {attempt}:')
        print(f'  LLM response: {response[:80]}')
        
        try:
            result = parse_with_pydantic(response)
            print(f'  VALIDATION PASSED: {result}')
            return result, attempt
        except (json.JSONDecodeError, ValidationError) as e:
            last_error = e
            error_msg = str(e)[:100]
            print(f'  VALIDATION FAILED: {error_msg}')
            if attempt < max_retries:
                print(f'  Sending error back to LLM and retrying...')
                # In real instructor: formats error as retry prompt:
                # "Your previous response failed validation: {error}. Please fix it."
    
    raise ValidationError.from_exception_data(
        'ProductReview',
        [{"type": "value_error", "msg": str(last_error), "loc": (), "input": {}}]
    ) if hasattr(ValidationError, 'from_exception_data') else last_error


# Scenario 1: First attempt succeeds
print('=== Scenario 1: First attempt succeeds ===')
mock_responses_1 = [
    '{"sentiment": "positive", "score": 8, "summary": "Great product"}',
]
result, attempts = simulate_instructor_retry(mock_responses_1, ProductReview)
print(f'\nSuccess in {attempts} attempt(s).')

# Scenario 2: Fails then succeeds (the common retry case)
print('\n=== Scenario 2: Fails once then succeeds ===')
mock_responses_2 = [
    'Here is the analysis: {"sentiment": "positive", "score": "eight", "summary": "Great product"}',  # score wrong type
    '{"sentiment": "positive", "score": 8, "summary": "Great product"}',  # fixed
]
result, attempts = simulate_instructor_retry(mock_responses_2, ProductReview)
print(f'\nSuccess in {attempts} attempt(s).')

=== Scenario 1: First attempt succeeds ===

Attempt 1:
  LLM response: {"sentiment": "positive", "score": 8, "summary": "Great product"}
  VALIDATION PASSED: sentiment=<SentimentEnum.positive: 'positive'> score=8 summary='Great product'

Success in 1 attempt(s).

=== Scenario 2: Fails once then succeeds ===

Attempt 1:
  LLM response: Here is the analysis: {"sentiment": "positive", "score": "eight", "summary": "Gr
  VALIDATION FAILED: 1 validation error for ProductReview
score
  Input should be a valid integer, unable to parse string
  Sending error back to LLM and retrying...

Attempt 2:
  LLM response: {"sentiment": "positive", "score": 8, "summary": "Great product"}
  VALIDATION PASSED: sentiment=<SentimentEnum.positive: 'positive'> score=8 summary='Great product'

Success in 2 attempt(s).


In [9]:
# Nested Pydantic models for complex outputs

class OrderItem(BaseModel):
    name: str
    quantity: int
    unit_price: float
    
    @field_validator('quantity')
    @classmethod
    def quantity_positive(cls, v):
        if v <= 0:
            raise ValueError('quantity must be positive')
        return v


class Order(BaseModel):
    order_id: str
    customer_name: str
    items: list[OrderItem]
    total: float
    currency: str = 'USD'
    
    @field_validator('total')
    @classmethod
    def total_must_be_positive(cls, v):
        if v < 0:
            raise ValueError('total cannot be negative')
        return round(v, 2)


# Simulate instructor extracting a nested Order from natural language
MOCK_ORDER_RESPONSE = '''
{
  "order_id": "ORD-2024-001",
  "customer_name": "Jane Smith",
  "items": [
    {"name": "Widget A", "quantity": 3, "unit_price": 9.99},
    {"name": "Gadget B", "quantity": 1, "unit_price": 49.99}
  ],
  "total": 79.96,
  "currency": "USD"
}
'''

order_data = json.loads(MOCK_ORDER_RESPONSE.strip())
order = Order(**order_data)

print('Parsed nested Order object:')
print(f'  Order ID: {order.order_id}')
print(f'  Customer: {order.customer_name}')
print(f'  Items: {len(order.items)}')
for item in order.items:
    print(f'    - {item.name}: {item.quantity}x ${item.unit_price}')
print(f'  Total: ${order.total} {order.currency}')
print()
print('With instructor, you would write:')
print('  order = client.messages.create(response_model=Order, ...)')
print('  # order is already an Order instance -- no json.loads(), no **unpacking')

Parsed nested Order object:
  Order ID: ORD-2024-001
  Customer: Jane Smith
  Items: 2
    - Widget A: 3x $9.99
    - Gadget B: 1x $49.99
  Total: $79.96 USD

With instructor, you would write:
  order = client.messages.create(response_model=Order, ...)
  # order is already an Order instance -- no json.loads(), no **unpacking


## Section 4: OpenAI JSON Mode

OpenAI's API provides `response_format={"type": "json_object"}` which guarantees the response is valid JSON -- but does NOT guarantee it matches your schema.

In [10]:
# OpenAI JSON mode -- production code pattern
#
# from openai import OpenAI
# client = OpenAI()
#
# response = client.chat.completions.create(
#     model='gpt-4o',
#     response_format={'type': 'json_object'},  # Guarantees valid JSON
#     messages=[
#         {'role': 'system', 'content': 'Return JSON matching: {"sentiment": str, "score": int, "summary": str}'},
#         {'role': 'user', 'content': 'Analyze: "Best purchase ever, would buy again!"'}
#     ]
# )
#
# raw_json = json.loads(response.choices[0].message.content)  # Always valid JSON
# # BUT: you still need Pydantic to validate the schema:
# review = ProductReview(**raw_json)  # May raise ValidationError

# Simulate the OpenAI JSON mode output
SIMULATED_OPENAI_JSON_MODE_RESPONSES = [
    # Correct schema
    '{"sentiment": "positive", "score": 9, "summary": "Best purchase ever"}',
    # Valid JSON but wrong schema (JSON mode does not prevent this)
    '{"feeling": "good", "rating": 9, "notes": "Best purchase ever"}',
    # Valid JSON, wrong type
    '{"sentiment": "positive", "score": "nine", "summary": "Best purchase ever"}',
]

print('OpenAI JSON mode: guarantees valid JSON, NOT schema compliance')
print()
for i, response in enumerate(SIMULATED_OPENAI_JSON_MODE_RESPONSES):
    print(f'Response {i+1}: {response}')
    try:
        data = json.loads(response)  # JSON mode guarantees this works
        review = ProductReview(**data)  # But schema validation may still fail
        print(f'  Schema validation: PASSED -> {review.sentiment.value}, score={review.score}')
    except ValidationError as e:
        print(f'  Schema validation: FAILED -> {e.errors()[0]["msg"]}')
    print()

print('Lesson: Always add Pydantic validation even when using JSON mode.')
print('instructor does this automatically and adds retry logic on top.')

OpenAI JSON mode: guarantees valid JSON, NOT schema compliance

Response 1: {"sentiment": "positive", "score": 9, "summary": "Best purchase ever"}
  Schema validation: PASSED -> positive, score=9

Response 2: {"feeling": "good", "rating": 9, "notes": "Best purchase ever"}
  Schema validation: FAILED -> Field required

Response 3: {"sentiment": "positive", "score": "nine", "summary": "Best purchase ever"}
  Schema validation: FAILED -> Input should be a valid integer, unable to parse string as an integer

Lesson: Always add Pydantic validation even when using JSON mode.
instructor does this automatically and adds retry logic on top.


## Section 5: Anthropic Tool Use for Structured Output

Anthropic's API supports tool use (function calling), which can be repurposed as a structured output mechanism. By defining a "tool" with your desired schema, you force the model to output schema-compliant parameters.

In [11]:
# Define a tool schema as the output format
# The 'tool' is not actually called -- it is a schema enforcement trick

REVIEW_TOOL_SCHEMA = {
    'name': 'record_review_analysis',
    'description': 'Record the structured analysis of a product review.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'sentiment': {
                'type': 'string',
                'enum': ['positive', 'negative', 'neutral'],
                'description': 'Overall sentiment of the review'
            },
            'score': {
                'type': 'integer',
                'minimum': 1,
                'maximum': 10,
                'description': 'Sentiment score from 1 (most negative) to 10 (most positive)'
            },
            'summary': {
                'type': 'string',
                'description': 'One-sentence summary of the review'
            }
        },
        'required': ['sentiment', 'score', 'summary']
    }
}

# Production usage pattern:
#
# import anthropic
# client = anthropic.Anthropic()
#
# response = client.messages.create(
#     model='claude-3-5-sonnet-20241022',
#     max_tokens=500,
#     tools=[REVIEW_TOOL_SCHEMA],
#     tool_choice={'type': 'tool', 'name': 'record_review_analysis'},  # Force this tool
#     messages=[{
#         'role': 'user',
#         'content': 'Analyze this review: "Fantastic product, exceeded expectations!"'
#     }]
# )
#
# # Extract the tool_use block
# tool_use = next(b for b in response.content if b.type == 'tool_use')
# structured_output = tool_use.input  # Already a dict, no JSON parsing needed
# review = ProductReview(**structured_output)  # Validate with Pydantic

# Simulate what the API returns
SIMULATED_TOOL_USE_RESPONSE = {
    'type': 'tool_use',
    'name': 'record_review_analysis',
    'input': {
        'sentiment': 'positive',
        'score': 9,
        'summary': 'Fantastic product that significantly exceeded customer expectations.'
    }
}

# The tool_use.input is already a dict -- the API handles JSON parsing
structured_output = SIMULATED_TOOL_USE_RESPONSE['input']
review = ProductReview(**structured_output)

print('Tool use structured output:')
print(f'  Sentiment: {review.sentiment.value}')
print(f'  Score: {review.score}/10')
print(f'  Summary: {review.summary}')
print()
print('Advantages of tool use approach:')
print('  - API enforces JSON structure (no markdown fences, no prose)')
print('  - tool_choice="required" means model MUST call the tool')
print('  - Schema defined in JSON Schema format (language-agnostic)')
print('  - No retry logic needed for formatting errors')
print()
print('instructor handles all of this automatically with the Pydantic model.')

Tool use structured output:
  Sentiment: positive
  Score: 9/10
  Summary: Fantastic product that significantly exceeded customer expectations.

Advantages of tool use approach:
  - API enforces JSON structure (no markdown fences, no prose)
  - tool_choice="required" means model MUST call the tool
  - Schema defined in JSON Schema format (language-agnostic)
  - No retry logic needed for formatting errors

instructor handles all of this automatically with the Pydantic model.


## Section 6: Real Use Cases

Structured outputs unlock reliable LLM integration in data pipelines, APIs, and automation workflows.

In [12]:
# Use Case 1: Information extraction from text
# Extract all entities from a contract or document

class ContractParty(BaseModel):
    name: str
    role: str  # 'buyer', 'seller', 'contractor', etc.


class ContractExtraction(BaseModel):
    parties: list[ContractParty]
    effective_date: Optional[str] = None
    expiration_date: Optional[str] = None
    total_value: Optional[float] = None
    currency: str = 'USD'
    key_obligations: list[str]
    governing_law: Optional[str] = None


SAMPLE_CONTRACT = """
SERVICE AGREEMENT
Effective Date: March 1, 2024
This Agreement is entered into between Acme Corp (hereinafter "Client") and 
TechServices LLC (hereinafter "Provider").

The Provider agrees to deliver software development services.
The Client agrees to pay $50,000 USD within 30 days of invoice.
This Agreement expires on December 31, 2024.
This Agreement shall be governed by the laws of California.
"""

# Simulated extraction result (what instructor would return)
MOCK_EXTRACTION_RESULT = ContractExtraction(
    parties=[
        ContractParty(name='Acme Corp', role='client'),
        ContractParty(name='TechServices LLC', role='provider'),
    ],
    effective_date='2024-03-01',
    expiration_date='2024-12-31',
    total_value=50000.0,
    currency='USD',
    key_obligations=[
        'Provider delivers software development services',
        'Client pays $50,000 within 30 days of invoice',
    ],
    governing_law='California',
)

print('Information Extraction Use Case')
print('Input: Contract text')
print('Output: Structured ContractExtraction object')
print()
print(f'Parties: {[(p.name, p.role) for p in MOCK_EXTRACTION_RESULT.parties]}')
print(f'Effective date: {MOCK_EXTRACTION_RESULT.effective_date}')
print(f'Value: ${MOCK_EXTRACTION_RESULT.total_value:,.2f} {MOCK_EXTRACTION_RESULT.currency}')
print(f'Obligations: {MOCK_EXTRACTION_RESULT.key_obligations}')
print(f'Governing law: {MOCK_EXTRACTION_RESULT.governing_law}')

Information Extraction Use Case
Input: Contract text
Output: Structured ContractExtraction object

Parties: [('Acme Corp', 'client'), ('TechServices LLC', 'provider')]
Effective date: 2024-03-01
Value: $50,000.00 USD
Obligations: ['Provider delivers software development services', 'Client pays $50,000 within 30 days of invoice']
Governing law: California


In [13]:
# Use Case 2: Classification with confidence score

class CategoryEnum(str, Enum):
    billing = 'billing'
    technical = 'technical'
    shipping = 'shipping'
    returns = 'returns'
    general = 'general'


class SupportTicketClassification(BaseModel):
    category: CategoryEnum
    confidence: float  # 0.0 to 1.0
    priority: int  # 1 (low) to 5 (urgent)
    requires_human: bool
    suggested_response_template: str
    
    @field_validator('confidence')
    @classmethod
    def confidence_in_range(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError('confidence must be between 0.0 and 1.0')
        return round(v, 3)
    
    @field_validator('priority')
    @classmethod
    def priority_in_range(cls, v):
        if not 1 <= v <= 5:
            raise ValueError('priority must be between 1 and 5')
        return v


# Simulate classifying support tickets
MOCK_TICKETS = [
    {
        'ticket': "I was charged twice for my order #12345!",
        'classification': SupportTicketClassification(
            category=CategoryEnum.billing,
            confidence=0.97,
            priority=4,
            requires_human=True,
            suggested_response_template='billing_dispute'
        )
    },
    {
        'ticket': "Where is my tracking number?",
        'classification': SupportTicketClassification(
            category=CategoryEnum.shipping,
            confidence=0.92,
            priority=2,
            requires_human=False,
            suggested_response_template='shipping_tracking_lookup'
        )
    },
]

print('Support Ticket Classification Use Case')
print()
for item in MOCK_TICKETS:
    c = item['classification']
    print(f'Ticket: "{item["ticket"]}"')
    print(f'  Category: {c.category.value} (confidence: {c.confidence:.0%})')
    print(f'  Priority: {c.priority}/5, Requires human: {c.requires_human}')
    print(f'  Template: {c.suggested_response_template}')
    print()

Support Ticket Classification Use Case



Ticket: "I was charged twice for my order #12345!"
  Category: billing (confidence: 97%)
  Priority: 4/5, Requires human: True
  Template: billing_dispute

Ticket: "Where is my tracking number?"
  Category: shipping (confidence: 92%)
  Priority: 2/5, Requires human: False
  Template: shipping_tracking_lookup



In [14]:
# Use Case 3: Data transformation pipeline
# Convert unstructured product descriptions to structured database records

class ProductDimensions(BaseModel):
    width_cm: Optional[float] = None
    height_cm: Optional[float] = None
    depth_cm: Optional[float] = None
    weight_kg: Optional[float] = None


class ProductRecord(BaseModel):
    name: str
    sku: Optional[str] = None
    price_usd: float
    category: str
    features: list[str]
    dimensions: Optional[ProductDimensions] = None
    in_stock: bool = True


UNSTRUCTURED_PRODUCT_TEXT = """
The UltraBook Pro X15 is our flagship 15-inch laptop. Starting at $1,499. 
Comes with 32GB DDR5 RAM and a 1TB NVMe SSD. Weighs just 1.6kg.
Dimensions: 35cm x 23cm x 1.8cm. SKU: UBP-X15-32-1T.
Currently in stock. Category: Laptops.
"""

# Simulated instructor output
MOCK_PRODUCT_RECORD = ProductRecord(
    name='UltraBook Pro X15',
    sku='UBP-X15-32-1T',
    price_usd=1499.0,
    category='Laptops',
    features=['32GB DDR5 RAM', '1TB NVMe SSD', '15-inch display'],
    dimensions=ProductDimensions(
        width_cm=35.0,
        height_cm=23.0,
        depth_cm=1.8,
        weight_kg=1.6
    ),
    in_stock=True,
)

print('Data Transformation Pipeline Use Case')
print('Input: Unstructured product description')
print(UNSTRUCTURED_PRODUCT_TEXT)
print('Output: Structured ProductRecord (ready for database insert)')
print()
print(json.dumps(MOCK_PRODUCT_RECORD.model_dump(), indent=2))

Data Transformation Pipeline Use Case
Input: Unstructured product description

The UltraBook Pro X15 is our flagship 15-inch laptop. Starting at $1,499. 
Comes with 32GB DDR5 RAM and a 1TB NVMe SSD. Weighs just 1.6kg.
Dimensions: 35cm x 23cm x 1.8cm. SKU: UBP-X15-32-1T.
Currently in stock. Category: Laptops.

Output: Structured ProductRecord (ready for database insert)

{
  "name": "UltraBook Pro X15",
  "sku": "UBP-X15-32-1T",
  "price_usd": 1499.0,
  "category": "Laptops",
  "features": [
    "32GB DDR5 RAM",
    "1TB NVMe SSD",
    "15-inch display"
  ],
  "dimensions": {
    "width_cm": 35.0,
    "height_cm": 23.0,
    "depth_cm": 1.8,
    "weight_kg": 1.6
  },
  "in_stock": true
}


## Section 7: Error Handling and Retries

Even with instructor, you need a strategy for what happens when validation fails after all retries are exhausted.

In [15]:
# Full error handling pattern for production

from pydantic import ValidationError

class StructuredOutputError(Exception):
    """Raised when structured output cannot be obtained after max retries."""
    def __init__(self, message: str, attempts: int, last_error: Exception):
        self.attempts = attempts
        self.last_error = last_error
        super().__init__(f'{message} (after {attempts} attempts): {last_error}')


def get_structured_output_with_retry(
    mock_responses: list[str],
    model_class: type[BaseModel],
    max_retries: int = 3,
    fallback_value = None,
):
    """
    Get structured output with retry and graceful fallback.
    
    In production with instructor:
        result = client.messages.create(
            response_model=ModelClass,
            max_retries=3,  # instructor handles retries internally
            ...
        )
    
    This function demonstrates the same logic manually.
    """
    last_error = None
    
    for attempt in range(1, max_retries + 1):
        response = mock_responses[min(attempt - 1, len(mock_responses) - 1)]
        
        try:
            result = parse_with_pydantic(response)
            print(f'  Succeeded on attempt {attempt}')
            return result
        except (json.JSONDecodeError, ValidationError, KeyError) as e:
            last_error = e
            print(f'  Attempt {attempt} failed: {type(e).__name__}: {str(e)[:60]}')
            # In production: format error and send back to LLM for retry
    
    if fallback_value is not None:
        print(f'  All {max_retries} attempts failed. Using fallback value.')
        return fallback_value
    
    raise StructuredOutputError(
        f'Failed to get valid {model_class.__name__}',
        attempts=max_retries,
        last_error=last_error
    )


# Test: eventually succeeds
print('Test 1: Eventually succeeds')
responses_eventually_good = [
    'not json at all',
    '{"sentiment": "positive", "score": "bad_type", "summary": "ok"}',
    '{"sentiment": "positive", "score": 8, "summary": "Great product"}',
]
result = get_structured_output_with_retry(responses_eventually_good, ProductReview)
print(f'Result: {result}')

# Test: all attempts fail, use fallback
print('\nTest 2: All attempts fail, fallback used')
responses_always_bad = ['not json', 'still not json', 'never json']
fallback = ProductReview(sentiment='neutral', score=5, summary='Unable to analyze')
result = get_structured_output_with_retry(responses_always_bad, ProductReview, fallback_value=fallback)
print(f'Fallback result: {result}')

# Test: all attempts fail, no fallback, raise error
print('\nTest 3: All attempts fail, no fallback, exception raised')
try:
    result = get_structured_output_with_retry(responses_always_bad, ProductReview)
except StructuredOutputError as e:
    print(f'StructuredOutputError raised: {str(e)[:80]}')

Test 1: Eventually succeeds
  Attempt 1 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  Attempt 2 failed: ValidationError: 1 validation error for ProductReview
score
  Input should be
  Succeeded on attempt 3
Result: sentiment=<SentimentEnum.positive: 'positive'> score=8 summary='Great product'

Test 2: All attempts fail, fallback used
  Attempt 1 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  Attempt 2 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  Attempt 3 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  All 3 attempts failed. Using fallback value.
Fallback result: sentiment=<SentimentEnum.neutral: 'neutral'> score=5 summary='Unable to analyze'

Test 3: All attempts fail, no fallback, exception raised
  Attempt 1 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  Attempt 2 failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  Attempt 3 failed: JSONDecodeError: Expect

In [16]:
# The retry prompt template -- what instructor sends back to the model

def format_retry_prompt(original_prompt: str, bad_response: str, validation_error: str) -> str:
    """Format a retry prompt that includes the validation error as feedback."""
    return f"""Your previous response did not pass validation.

Original request:
{original_prompt}

Your response:
{bad_response}

Validation error:
{validation_error}

Please fix the error and return the correct JSON format.
Do not include any text outside the JSON object."""


# Example retry prompt
bad_response = '{"sentiment": "positive", "score": "eight", "summary": ""}'
try:
    parse_with_pydantic(bad_response)
except (ValidationError, json.JSONDecodeError) as e:
    retry_prompt = format_retry_prompt(
        original_prompt='Analyze this review: "Best laptop ever!"',
        bad_response=bad_response,
        validation_error=str(e)[:200]
    )
    print('Retry prompt sent to model:')
    print(retry_prompt)

Retry prompt sent to model:
Your previous response did not pass validation.

Original request:
Analyze this review: "Best laptop ever!"

Your response:
{"sentiment": "positive", "score": "eight", "summary": ""}

Validation error:
2 validation errors for ProductReview
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='eight', input_type=str]
    For further information 

Please fix the error and return the correct JSON format.
Do not include any text outside the JSON object.


## Comparison: When to Use Each Approach

| Approach | Reliability | Effort | Best For |
|----------|------------|--------|-----------|
| Prompt + json.loads | Low | Minimal | Prototypes only |
| Pydantic validation | Medium | Low | Simple schemas, single model |
| instructor | High | Low | Production, complex schemas |
| Anthropic tool use | High | Medium | Language-agnostic, API-enforced |
| OpenAI JSON mode | Medium | Low | Guarantees valid JSON, not schema |

## Key Takeaways

1. **Never trust raw LLM output**: LLMs return markdown fences, prose, wrong types, and missing fields. Always validate.

2. **Pydantic is your schema contract**: Define the expected output as a Pydantic model. This serves as documentation, validation, and type safety in one.

3. **instructor = Pydantic + automatic retries**: The library handles parsing, validation, and retry logic. You just pass a response_model and receive a typed Python object.

4. **Tool use provides API-level guarantees**: Anthropic tool use and OpenAI function calling force schema compliance at the API level, eliminating most parsing errors.

5. **Always validate at system boundaries**: Even if the LLM returns correct output 99% of the time, the 1% of failures will cause production incidents. Validate every response.

6. **Have a fallback strategy**: Define what happens when validation fails after max retries. Options: return a default value, queue for human review, or raise an error that triggers a circuit breaker.

7. **Structured outputs enable data pipelines**: Reliable schema compliance is what makes it possible to feed LLM outputs directly into databases, APIs, and downstream systems without human oversight of every record.